# 01 — Bronze
Lands the raw H&M CSVs as Delta tables, unchanged.

Every column is read with an **explicit schema** rather than `inferSchema`. Inference
would read `article_id` (`0663713001`) as a number and destroy the leading zero, and it
costs an extra pass over 3.5 GB to guess what we already know.

Bronze does no cleaning. `t_dat` stays a string here and is parsed in silver, so that the
bronze table is a faithful copy of the file and any unparseable date is visible rather
than silently null.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

CATALOG = "workspace"
SCHEMA = "retail"
VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/raw"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

DataFrame[]

## Transactions

In [0]:
transactions_schema = StructType([
    StructField("t_dat", StringType(), nullable=True),            # 'YYYY-MM-DD'; parsed in silver
    StructField("customer_id", StringType(), nullable=True),      # 64-char hex
    StructField("article_id", StringType(), nullable=True),       # keeps leading zeros
    StructField("price", DoubleType(), nullable=True),            # H&M-scaled, not currency
    StructField("sales_channel_id", IntegerType(), nullable=True) # 1 = store, 2 = online
])

transactions = (
    spark.read
    .schema(transactions_schema)
    .option("header", True)
    .csv(f"{VOLUME}/transactions_train.csv")
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

(transactions.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze_transactions"))

## Articles

In [0]:
articles_schema = StructType([
    StructField("article_id", StringType(), nullable=True),
    StructField("product_code", StringType(), nullable=True),
    StructField("prod_name", StringType(), nullable=True),
    StructField("product_type_no", IntegerType(), nullable=True),
    StructField("product_type_name", StringType(), nullable=True),
    StructField("product_group_name", StringType(), nullable=True),
    StructField("graphical_appearance_no", IntegerType(), nullable=True),
    StructField("graphical_appearance_name", StringType(), nullable=True),
    StructField("colour_group_code", StringType(), nullable=True),
    StructField("colour_group_name", StringType(), nullable=True),
    StructField("perceived_colour_value_id", IntegerType(), nullable=True),
    StructField("perceived_colour_value_name", StringType(), nullable=True),
    StructField("perceived_colour_master_id", IntegerType(), nullable=True),
    StructField("perceived_colour_master_name", StringType(), nullable=True),
    StructField("department_no", IntegerType(), nullable=True),
    StructField("department_name", StringType(), nullable=True),
    StructField("index_code", StringType(), nullable=True),
    StructField("index_name", StringType(), nullable=True),
    StructField("index_group_no", IntegerType(), nullable=True),
    StructField("index_group_name", StringType(), nullable=True),
    StructField("section_no", IntegerType(), nullable=True),
    StructField("section_name", StringType(), nullable=True),
    StructField("garment_group_no", IntegerType(), nullable=True),
    StructField("garment_group_name", StringType(), nullable=True),
    StructField("detail_desc", StringType(), nullable=True)       # free text; the retrieval corpus
])

articles = (
    spark.read
    .schema(articles_schema)
    .option("header", True)
    .option("multiLine", True)      # some detail_desc values contain newlines
    .option("quote", '"')
    .option("escape", '"')
    .csv(f"{VOLUME}/articles.csv")
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

(articles.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze_articles"))

## Row counts
These numbers go in the README. They are read back from the written tables, not from the
DataFrames above, so they describe what is actually stored.

In [0]:
for table in ["bronze_transactions", "bronze_articles"]:
    print(f"{CATALOG}.{SCHEMA}.{table}: {spark.table(table).count():,} rows")

workspace.retail.bronze_transactions: 31,788,324 rows
workspace.retail.bronze_articles: 105,542 rows


In [0]:
display(spark.table("bronze_transactions").limit(5))

t_dat,customer_id,article_id,price,sales_channel_id,_source_file,_ingested_at
2019-05-29,a210347cacf364ca6b7855652081a84147d3f16dcdb109a7d9561eacb518f399,0733438001,0.033881355932203386,2,dbfs:/Volumes/workspace/retail/raw/transactions_train.csv,2026-10-13T09:44:11.776Z
2019-05-29,a212dfff22fe06b02d720b45ff68191ce684d892e688aa07042028b89525a2fa,0781613001,0.050830508474576264,2,dbfs:/Volumes/workspace/retail/raw/transactions_train.csv,2026-10-13T09:44:11.776Z
2019-05-29,a214486b31cab3ae8d4f177a15203b7f557658045a78a2761e65ceb256a15b02,0705416001,0.016932203389830508,1,dbfs:/Volumes/workspace/retail/raw/transactions_train.csv,2026-10-13T09:44:11.776Z
2019-05-29,a2167773d4f34255331cdc74db82857f012b2fa6704d9b4c0fc783a317831ba9,0747939002,0.013542372881355931,2,dbfs:/Volumes/workspace/retail/raw/transactions_train.csv,2026-10-13T09:44:11.776Z
2019-05-29,a2167773d4f34255331cdc74db82857f012b2fa6704d9b4c0fc783a317831ba9,0723529001,0.020322033898305086,2,dbfs:/Volumes/workspace/retail/raw/transactions_train.csv,2026-10-13T09:44:11.776Z


In [0]:
display(spark.table("bronze_articles").limit(5))

article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc,_source_file,_ingested_at
0108775015,0108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,09,Black,4,Dark,5,Black,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,dbfs:/Volumes/workspace/retail/raw/articles.csv,2026-10-13T09:44:47.415Z
0108775044,0108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,dbfs:/Volumes/workspace/retail/raw/articles.csv,2026-10-13T09:44:47.415Z
0108775051,0108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,1,Dusty Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,dbfs:/Volumes/workspace/retail/raw/articles.csv,2026-10-13T09:44:47.415Z
0110065001,0110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,09,Black,4,Dark,5,Black,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and provide good support. Narrow adjustable shoulder straps and a narrow hook-and-eye fastening at the back. Without visible seams for greater comfort.",dbfs:/Volumes/workspace/retail/raw/articles.csv,2026-10-13T09:44:47.415Z
0110065002,0110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,3,Light,9,White,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and provide good support. Narrow adjustable shoulder straps and a narrow hook-and-eye fastening at the back. Without visible seams for greater comfort.",dbfs:/Volumes/workspace/retail/raw/articles.csv,2026-10-13T09:44:47.415Z
